In [13]:
import os
import torch
from transformers import HiggsAudioV2TokenizerModel, AutoFeatureExtractor
from torch.export import Dim

### Load Model

In [2]:
def _resolve_model_path(name_or_path: str) -> str:
    if os.path.isdir(name_or_path):
        return name_or_path
    from huggingface_hub import snapshot_download

    return snapshot_download(name_or_path)

In [3]:
model_path = _resolve_model_path("k2-fsa/OmniVoice")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [4]:
audio_tokenizer_path = model_path + "/audio_tokenizer"

In [11]:
ENCODER_PATH = "onnx_higgs_audio_v2_tokenizer_encoder"
DECODER_PATH = "onnx_higgs_audio_v2_tokenizer_decoder"

In [12]:
os.makedirs(ENCODER_PATH, exist_ok=True)
os.makedirs(DECODER_PATH, exist_ok=True)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
model = HiggsAudioV2TokenizerModel.from_pretrained(
    audio_tokenizer_path,
    torch_dtype=torch.float32,
).to(device).eval()

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

In [8]:
feature_extractor = AutoFeatureExtractor.from_pretrained(audio_tokenizer_path)

### Encoder

In [9]:
dummy_audio = torch.randn(1, 1, 24000, device=device)

In [10]:
class EncoderWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_values):
        outputs = self.model.encode(input_values)
        return outputs.audio_codes

encoder = EncoderWrapper(model).eval()

In [22]:
torch.onnx.export(
    encoder,
    (dummy_audio,),
    os.path.join(ENCODER_PATH, "model.onnx"),
    input_names=["input_values"],
    output_names=["audio_codes"],
    dynamic_axes={
        "input_values": {0: "batch_size", 2: "audio_length"},
        "audio_codes": {0: "batch_size", 2: "codes_length"},
    },
    opset_version=14,          
    do_constant_folding=False,
    dynamo=False,              
    external_data=False,
)

/tmp/ipykernel_45405/1339114019.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


### Decoder

In [23]:
with torch.no_grad():
    dummy_codes = encoder(dummy_audio)

In [24]:
dummy_codes.shape

torch.Size([1, 8, 25])

In [25]:
class DecoderWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, audio_codes):
        outputs = self.model.decode(audio_codes)
        return outputs.audio_values

decoder = DecoderWrapper(model).eval()

In [26]:
torch.onnx.export(
    decoder,
    (dummy_codes,),
    os.path.join("onnx_higgs_audio_v2_tokenizer_decoder", "model.onnx"),
    input_names=["audio_codes"],
    output_names=["audio_values"],
    dynamic_shapes = (
        {
            0: Dim("batch_size", min=1),
            2: Dim("codes_length", min=1),
        },
    ),
    opset_version=18,
    do_constant_folding=False,
    dynamo=True,
    external_data=False,
)

W1101 20:55:48.252000 45405 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1101 20:55:48.254000 45405 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W1101 20:55:48.255000 45405 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool
W1101 20:55:48.257000 45405 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d


[torch.onnx] Obtain model graph for `DecoderWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DecoderWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.14.1+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"audio_codes"<INT64,[batch_size,8,codes_length]>
            ),
            outputs=(
                %"audio_values"<FLOAT,[batch_size,1,960*codes_length]>
            ),
            initializers=(
                %"model.acoustic_decoder.conv1.weight"<FLOAT,[1024,256,7]>{TorchTensor(...)},
                %"model.acoustic_decoder.conv1.bias"<FLOAT,[1024]>{TorchTensor(...)},
                %"model.acoustic_decoder.block.0.snake1.alpha"<FLOAT,[1,1024,1]>{TorchTensor(...)},
                %"model.acoustic_decoder.block.0.conv_t1.weight"<FLOAT,[1024,512,16]>{TorchTensor(...)},
                %"model.acoustic_decoder.block.0.conv_t1.bias"<FLOAT,[512]>{TorchTensor(